In [2]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from qiskit_machine_learning.algorithms import QSVC

In [3]:
data = load_breast_cancer()

In [8]:
X = data.data      
y = data.target 

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [10]:
len(X_train), len(X_test)

(455, 114)

In [12]:
n_qubits = 4

scaler = StandardScaler().fit(X_train)
pca = PCA(n_components=n_qubits).fit(scaler.transform(X_train))
to_angle = MinMaxScaler(feature_range=(0, np.pi)).fit(pca.transform(scaler.transform(X_train)))

def prepare(X):
    return to_angle.transform(pca.transform(scaler.transform(X)))

X_train_q = prepare(X_train)
X_test_q = prepare(X_test)

In [13]:
classical = SVC(kernel="rbf")
classical.fit(X_train_q, y_train)
acc_classical = accuracy_score(y_test, classical.predict(X_test_q))
print("Classical SVM accuracy:", round(acc_classical, 3))

Classical SVM accuracy: 0.956


In [14]:
x = ParameterVector("x", n_qubits)
circuit = QuantumCircuit(n_qubits)

for i in range(n_qubits):
    circuit.ry(x[i], i)  

quantum = QSVC(quantum_kernel=FidelityStatevectorKernel(feature_map=circuit))
quantum.fit(X_train_q, y_train)
acc_quantum = accuracy_score(y_test, quantum.predict(X_test_q))
print("Quantum SVM accuracy:", round(acc_quantum, 3))


Quantum SVM accuracy: 0.956


In [15]:
entangled_circuit = zz_feature_map(n_qubits, reps=2)

quantum_zz = QSVC(quantum_kernel=FidelityStatevectorKernel(feature_map=entangled_circuit))
quantum_zz.fit(X_train_q, y_train)
acc_quantum_zz = accuracy_score(y_test, quantum_zz.predict(X_test_q))
print("Quantum SVM (entangled) accuracy:", round(acc_quantum_zz, 3))

Quantum SVM (entangled) accuracy: 0.789
